# RQ1 association

Kendall's tau-b between the six VBench 1.0 scores and clip-level mean human enjoyment.

The confirmatory question is pairwise: when one clip has a higher dimension score than another, does it also tend to have higher enjoyment? Tau-b summarizes concordant pairs minus discordant pairs and adjusts for ties. The coefficient is `(P - Q) / sqrt((P + Q + T)(P + Q + U))`, from `scipy.stats.kendalltau(..., variant="b")`. P and Q are concordant and discordant clip pairs. T counts pairs tied only on the dimension score. U counts pairs tied only on mean enjoyment. Pairs tied on both are in neither count. With no ties this equals tau-a. Positive tau-b means higher scores tend to accompany higher enjoyment. The pairs are not extra independent clips. N is the number of clips.

Spearman is not estimated. Tau-b and Spearman are different scales and are not compared. A constant column makes tau-b undefined. That dimension stays in the six-test Holm family as an internal p of 1, and its coefficient and p-values are reported as unavailable rather than as zero.

Reads `data/metric_scores.csv`, keeping `source_id` as text so values such as `001` and `1` stay distinct. When any `source_id` is blank, fills only those blanks from `data/clip_manifest.csv` after rejecting null, duplicate, or conflicting manifest IDs. Reads `data/human_ratings.csv` when that file exists (`video_id`, `rater_id`, `enjoyment` as integers 1–5).

The confirmatory family is the six VBench 1.0 columns. Identity and clothes stay excluded. `vbench2_human_anatomy` is exploratory and is not in the family. `dynamic_degree` stays in the family; on this pilot it is constant, so tau-b is undefined.

This notebook does not score clips and does not invent ratings. If enjoyment ratings are missing, or confirmatory inference is blocked, it stops, writes no association claims, and removes a prior `rq1_kendall_results.csv` and any leftover `rq1_spearman_results.csv`. Non-numeric and infinite scores are rejected. Blank scores stay missing and are never filled with zero. Scores are not rounded or jittered before ties are judged. Clips with fewer than 3 ratings are excluded. Kendall's tau-b needs at least 4 eligible clips. Repeated `source_id` values, including provisional `unlabeled_pilot`, skip confirmatory inference until real drama IDs exist. `unlabeled_pilot` is not treated as a verified shared drama or as evidence that the clips are independent.

Bootstrap 95% percentile intervals resample paired clips with replacement. A replicate that makes either column constant is undefined: it is counted, omitted from the percentiles, and the notebook warns. The interval is reported only when at least 100 replicates are defined; otherwise the limits stay unavailable and the notebook warns. Intervals condition on the observed viewer panel and are not simultaneous family-wide intervals. The permutation test holds metric scores fixed and permutes mean enjoyment. Absolute tau-b is the two-sided statistic. Complete cases of size 7 or fewer (`7! = 5040`) use every enjoyment permutation. Larger samples use Monte Carlo. The Monte Carlo p-value is `(1 + count of |tau| at least the observed |tau|) / (1 + permutations)` and is never zero. Median enjoyment repeats tau-b as a sensitivity check, not as a second confirmatory test.

Defaults are seed 42, 2000 bootstrap replicates, and 4999 Monte Carlo permutations. Override them with `RQ1_SEED`, `RQ1_N_BOOTSTRAP`, and `RQ1_N_PERMUTATIONS`.


In [ ]:
from pathlib import Path
import os
from itertools import permutations
import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from statsmodels.stats.multitest import multipletests

DIMENSIONS = [
    "vbench_subject_consistency",
    "vbench_background_consistency",
    "vbench_motion_smoothness",
    "vbench_dynamic_degree",
    "vbench_aesthetic_quality",
    "vbench_imaging_quality",
]
SCALE_MIN, SCALE_MAX = 1, 5
MIN_RATERS = 3
MIN_CLIPS = 4
ALPHA = 0.05
DEFAULT_N_BOOTSTRAP = 2000
DEFAULT_N_PERMUTATIONS = 4999
DEFAULT_SEED = 42
# Percentile intervals need this many defined replicates. Fewer stay unavailable.
MIN_VALID_BOOTSTRAPS = 100
# 7! = 5040, next to the default Monte Carlo budget. 8! = 40320, so larger n is Monte Carlo.
EXACT_MAX_N = 7
PROVISIONAL_SOURCE = "unlabeled_pilot"
RESULTS_NAME = "rq1_kendall_results.csv"
LEGACY_RESULTS_NAME = "rq1_spearman_results.csv"


def configured_int(name, default, minimum):
    raw = os.environ.get(name)
    if raw is None or str(raw).strip() == "":
        return default
    value = int(raw)
    if value < minimum:
        raise ValueError(name + " must be >= " + str(minimum) + ".")
    return value


N_BOOTSTRAP = configured_int("RQ1_N_BOOTSTRAP", DEFAULT_N_BOOTSTRAP, 1)
N_PERMUTATIONS = configured_int("RQ1_N_PERMUTATIONS", DEFAULT_N_PERMUTATIONS, 1)
SEED = configured_int("RQ1_SEED", DEFAULT_SEED, 0)


def stop(*lines):
    for line in lines:
        print(line)
    print("No association claims were written.")


def clear_prior_results():
    folder = output_dir()
    for name in (RESULTS_NAME, LEGACY_RESULTS_NAME):
        path = folder / name
        if path.is_file():
            path.unlink()
            print("Removed prior " + path.name + ".")


def repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / "data" / "metric_scores.csv").is_file():
            return candidate
    return cwd


def data_dir():
    override = os.environ.get("RQ1_DATA_DIR")
    if override:
        return Path(override).resolve()
    return repo_root() / "data"


def output_dir():
    override = os.environ.get("RQ1_OUT_DIR")
    if override:
        return Path(override).resolve()
    return repo_root() / "rq1_outputs"


def kendall_tau_b(x, y):
    result = kendalltau(np.asarray(x, float), np.asarray(y, float), variant="b")
    stat = getattr(result, "statistic", None)
    if stat is None:
        stat = result.correlation
    return float(stat)


def permutation_test(x, y, observed, rng):
    # Two-sided. Metric scores stay fixed; enjoyment is permuted.
    # The 1e-12 tolerance only absorbs floating-point noise around equality.
    threshold = abs(observed) - 1e-12
    if len(y) <= EXACT_MAX_N:
        extreme = 0
        total = 0
        for perm in permutations(np.asarray(y, float).tolist()):
            total += 1
            extreme += abs(kendall_tau_b(x, perm)) >= threshold
        return extreme / total, "exact"
    extreme = 0
    for _ in range(N_PERMUTATIONS):
        extreme += abs(kendall_tau_b(x, rng.permutation(y))) >= threshold
    return (extreme + 1) / (N_PERMUTATIONS + 1), "monte_carlo"


def clean_id(series):
    text = series.astype("string").str.strip()
    return text.mask(series.isna() | text.isna() | text.eq(""))


def validated_manifest(manifest):
    if not {"video_id", "source_id"}.issubset(manifest.columns):
        raise ValueError("clip_manifest.csv needs video_id and source_id columns.")
    video = clean_id(manifest["video_id"])
    source = clean_id(manifest["source_id"])
    if video.isna().any():
        raise ValueError("clip_manifest.csv has a null video_id.")
    if video.duplicated().any():
        pairs = pd.DataFrame({
            "video_id": video.astype(str),
            "source_id": source.astype("string"),
        })
        nunique = pairs.groupby("video_id", dropna=False)["source_id"].nunique(dropna=False)
        if nunique.gt(1).any():
            raise ValueError(
                "clip_manifest.csv has conflicting source_id values for the same video_id."
            )
        raise ValueError("clip_manifest.csv has duplicate video_id values.")
    if source.isna().any():
        raise ValueError("clip_manifest.csv has a null source_id.")
    return pd.DataFrame({"video_id": video.astype(str), "source_id": source.astype(str)})


def fill_source_ids(metrics, folder):
    metrics = metrics.copy()
    if "source_id" not in metrics.columns:
        metrics["source_id"] = pd.Series(pd.NA, index=metrics.index, dtype="string")
    metrics["source_id"] = clean_id(metrics["source_id"])
    missing = metrics["source_id"].isna()
    if not missing.any():
        return metrics
    manifest_path = folder / "clip_manifest.csv"
    if not manifest_path.is_file():
        return metrics
    source = validated_manifest(pd.read_csv(manifest_path, dtype=str))
    known = metrics.loc[~missing, ["video_id", "source_id"]].merge(
        source, on="video_id", how="left", suffixes=("_metric", "_manifest")
    )
    if len(known):
        conflict = known["source_id_manifest"].notna() & known["source_id_metric"].ne(
            known["source_id_manifest"]
        )
        if conflict.any():
            ids = ", ".join(sorted(known.loc[conflict, "video_id"].astype(str)))
            raise ValueError(
                "clip_manifest.csv source_id conflicts with metric_scores.csv for: " + ids
            )
    filled = metrics.merge(source, on="video_id", how="left", suffixes=("", "_from_manifest"))
    take = filled["source_id"].isna() & filled["source_id_from_manifest"].notna()
    filled.loc[take, "source_id"] = filled.loc[take, "source_id_from_manifest"]
    print(
        "Filled "
        + str(int(take.sum()))
        + " missing source_id values from "
        + manifest_path.name
        + "."
    )
    return filled.drop(columns=["source_id_from_manifest"])


def load_metrics(folder):
    path = folder / "metric_scores.csv"
    if not path.is_file():
        stop(
            "Missing " + str(path) + ".",
            "This notebook only reads CSVs. It does not score clips.",
        )
        return None
    metrics = pd.read_csv(path, dtype={"video_id": str, "source_id": str})
    if "video_id" not in metrics.columns:
        raise ValueError(str(path) + " needs a video_id column.")
    video = clean_id(metrics["video_id"])
    if video.isna().any() or video.duplicated().any():
        raise ValueError("metric_scores.csv needs one non-null video_id per clip.")
    metrics = metrics.copy()
    metrics["video_id"] = video.astype(str)
    metrics = fill_source_ids(metrics, folder)
    print("Loaded " + path.name + ": " + str(len(metrics)) + " clips.")
    return metrics


def parse_score_column(series, name):
    text = series.astype("string").str.strip()
    blank = series.isna() | text.isna() | text.eq("")
    try:
        values = pd.to_numeric(text.mask(blank), errors="raise")
    except ValueError:
        raise ValueError(
            name + " has non-numeric scores. Blank cells stay missing."
        )
    if np.isinf(values.to_numpy(dtype=float)).any():
        raise ValueError(name + " contains infinite scores.")
    return values


def load_ratings(folder):
    path = folder / "human_ratings.csv"
    if not path.is_file():
        stop(
            "Missing " + str(path) + ".",
            "Expected columns: video_id, rater_id, enjoyment (integers 1-5).",
            "Enjoyment has not been collected. This notebook does not invent ratings.",
        )
        return None
    ratings = pd.read_csv(path, dtype={"video_id": str, "rater_id": str})
    if ratings.empty:
        stop(
            str(path) + " has no enjoyment ratings.",
            "Expected columns: video_id, rater_id, enjoyment (integers 1-5).",
            "Enjoyment has not been collected. This notebook does not invent ratings.",
        )
        return None
    missing_cols = {"video_id", "rater_id", "enjoyment"} - set(ratings.columns)
    if missing_cols:
        raise ValueError(
            "human_ratings.csv is missing columns: " + ", ".join(sorted(missing_cols))
        )
    if ratings["enjoyment"].dropna().empty:
        stop(
            str(path) + " has no enjoyment values.",
            "Expected integers 1-5. This notebook does not invent ratings.",
        )
        return None
    return ratings


def validate_ratings(ratings, metrics):
    if ratings[["video_id", "rater_id"]].isna().any().any():
        raise ValueError("video_id and rater_id must be filled on every rating row.")
    if ratings.duplicated(["video_id", "rater_id"]).any():
        raise ValueError("Duplicate video_id, rater_id rows. Resolve them before aggregating.")
    numeric = pd.to_numeric(ratings["enjoyment"], errors="coerce")
    invalid_text = ratings["enjoyment"].notna() & numeric.isna()
    present = numeric.dropna()
    non_integer = (present % 1) != 0
    out_of_range = ~present.between(SCALE_MIN, SCALE_MAX)
    if invalid_text.any() or non_integer.any() or out_of_range.any():
        raise ValueError(
            "enjoyment must be integers from 1 to 5, or blank if missing. "
            "This notebook does not impute ratings."
        )
    unknown = set(ratings["video_id"]) - set(metrics["video_id"])
    if unknown:
        raise ValueError(
            "Ratings for video_ids that are not in metric_scores.csv: "
            + ", ".join(sorted(unknown))
        )
    checked = ratings.copy()
    checked["enjoyment"] = numeric
    return checked


def clip_table(metrics, ratings):
    dropped = int(ratings["enjoyment"].isna().sum())
    valid = ratings.dropna(subset=["enjoyment"])
    summary = valid.groupby("video_id", as_index=False)["enjoyment"].agg(
        human_mean="mean",
        human_median="median",
        n_raters="count",
        human_sd="std",
    )
    clips = metrics.merge(summary, on="video_id", how="left", validate="one_to_one")
    clips["included"] = clips["n_raters"].fillna(0).ge(MIN_RATERS)
    print("Rating rows used: " + str(len(valid)))
    print("Distinct raters: " + str(valid["rater_id"].nunique()))
    print("Rating rows with missing enjoyment dropped: " + str(dropped))
    included = clips.loc[clips["included"]].copy()
    excluded = int((~clips["included"]).sum())
    print(
        "Eligible clips: "
        + str(len(included))
        + " (minimum "
        + str(MIN_RATERS)
        + " raters)"
    )
    print("Clips excluded for too few raters: " + str(excluded))
    if excluded:
        print(
            "Excluded video_id values: "
            + ", ".join(clips.loc[~clips["included"], "video_id"].astype(str))
        )
    if len(included):
        view_cols = ["video_id", "n_raters", "human_mean", "human_sd"]
        if "source_id" in included.columns:
            view_cols.insert(1, "source_id")
        view = included[view_cols].copy()
        view["n_raters"] = view["n_raters"].astype(int)
        view["human_mean"] = view["human_mean"].map(lambda v: format(v, ".3f"))
        view["human_sd"] = view["human_sd"].map(
            lambda v: "NA" if pd.isna(v) else format(v, ".3f")
        )
        print(view.to_string(index=False))
    return included


def confirmatory_block(included):
    if "source_id" not in included.columns:
        return (
            "Flag: no source_id column on the scored clips.",
            "Confirmatory inference stays off until real drama IDs exist.",
        )
    raw = included["source_id"]
    if raw.isna().any():
        return (
            "Flag: source_id is missing for at least one eligible clip.",
            "Confirmatory inference stays off until real drama IDs exist.",
        )
    sources = raw.astype(str).str.strip()
    if sources.eq("").any():
        return (
            "Flag: source_id is missing for at least one eligible clip.",
            "Confirmatory inference stays off until real drama IDs exist.",
        )
    n_pilot = int(sources.eq(PROVISIONAL_SOURCE).sum())
    if n_pilot:
        return (
            "Flag: source_id "
            + PROVISIONAL_SOURCE
            + " on "
            + str(n_pilot)
            + " of "
            + str(len(sources))
            + " eligible clips ("
            + str(int(sources.nunique()))
            + " distinct source_id value(s)).",
            "Drama grouping is unknown, so these clips are not an independent set.",
            "Confirmatory inference stays off until real drama IDs exist.",
        )
    if sources.duplicated().any():
        repeated = sorted(sources[sources.duplicated(keep=False)].unique())
        return (
            "Flag: repeated source_id values: " + ", ".join(repeated) + ".",
            "Confirmatory clip-level inference stays off until each eligible clip has its own drama ID, or the analysis is cluster-aware.",
        )
    return None


def association(x, y, seed):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    n_unique_x = int(np.unique(x).size) if len(x) else 0
    n_unique_y = int(np.unique(y).size) if len(y) else 0
    out = {
        "n_complete_cases": int(len(x)),
        "n_unique_metric_values": n_unique_x,
        "n_unique_enjoyment_values": n_unique_y,
        "kendall_tau_b": np.nan,
        "ci_95_lower": np.nan,
        "ci_95_upper": np.nan,
        "p_permutation_raw": np.nan,
        "permutation_method": "not_applicable",
        "bootstrap_attempted_replicates": 0,
        "bootstrap_valid_replicates": 0,
        "bootstrap_undefined_replicates": 0,
        "analysis_status": "untestable",
    }
    if len(x) == 0:
        out["analysis_status"] = "all_missing"
        return out
    if len(x) < MIN_CLIPS:
        out["analysis_status"] = "too_few_clips"
        return out
    if n_unique_x < 2 or n_unique_y < 2:
        out["analysis_status"] = "no_variation"
        return out
    rng = np.random.default_rng(seed)
    tau_b = kendall_tau_b(x, y)
    p_raw, method = permutation_test(x, y, tau_b, rng)
    valid = []
    undefined = 0
    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, len(x), len(x))
        if np.unique(x[idx]).size < 2 or np.unique(y[idx]).size < 2:
            undefined += 1
            continue
        valid.append(kendall_tau_b(x[idx], y[idx]))
    if len(valid) >= MIN_VALID_BOOTSTRAPS:
        ci_low, ci_high = np.quantile(np.asarray(valid, float), [0.025, 0.975])
    else:
        ci_low, ci_high = np.nan, np.nan
    out.update(
        kendall_tau_b=tau_b,
        ci_95_lower=float(ci_low),
        ci_95_upper=float(ci_high),
        p_permutation_raw=float(p_raw),
        permutation_method=method,
        bootstrap_attempted_replicates=N_BOOTSTRAP,
        bootstrap_valid_replicates=len(valid),
        bootstrap_undefined_replicates=undefined,
        analysis_status="tested",
    )
    return out


def median_tau_b(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    if len(x) < MIN_CLIPS or np.unique(x).size < 2 or np.unique(y).size < 2:
        return np.nan
    return kendall_tau_b(x, y)


def fmt(value, digits):
    if pd.isna(value):
        return "NA"
    return format(float(value), "." + str(digits) + "f")


def empty_result(dimension, n_eligible, status, n_missing):
    return {
        "dimension": dimension,
        "analysis_status": status,
        "n_eligible_clips": n_eligible,
        "n_complete_cases": 0,
        "n_missing_metric": n_missing,
        "n_unique_metric_values": 0,
        "n_unique_enjoyment_values": 0,
        "kendall_tau_b": np.nan,
        "ci_95_lower": np.nan,
        "ci_95_upper": np.nan,
        "p_permutation_raw": np.nan,
        "p_permutation_holm": np.nan,
        "permutation_method": "not_applicable",
        "bootstrap_attempted_replicates": 0,
        "bootstrap_valid_replicates": 0,
        "bootstrap_undefined_replicates": 0,
        "kendall_tau_b_median": np.nan,
        "reject_holm": False,
    }


def analyze(included):
    n_eligible = len(included)
    rows = []
    p_for_holm = []
    for i, dimension in enumerate(DIMENSIONS):
        if dimension not in included.columns:
            rows.append(empty_result(dimension, n_eligible, "missing_column", n_eligible))
            p_for_holm.append(1.0)
            continue
        x = parse_score_column(included[dimension], dimension)
        n_missing = int(x.isna().sum())
        result = association(x, included["human_mean"], SEED + i)
        result.update(
            dimension=dimension,
            n_missing_metric=n_missing,
            n_eligible_clips=n_eligible,
            kendall_tau_b_median=(
                median_tau_b(x, included["human_median"])
                if result["analysis_status"] == "tested"
                else np.nan
            ),
            p_permutation_holm=np.nan,
            reject_holm=False,
        )
        rows.append(result)
        # Untestable prespecified dimensions stay in the family as p = 1.
        # Their displayed p-values remain unavailable.
        p_for_holm.append(
            result["p_permutation_raw"] if result["analysis_status"] == "tested" else 1.0
        )
    _reject, adjusted, _a, _b = multipletests(p_for_holm, alpha=ALPHA, method="holm")
    for row, p_adj, reject in zip(rows, adjusted, _reject):
        if row["analysis_status"] == "tested":
            row["p_permutation_holm"] = float(p_adj)
            row["reject_holm"] = bool(reject)
    return pd.DataFrame(rows)


def print_results(results):
    n_eligible = int(results["n_eligible_clips"].iloc[0]) if len(results) else 0
    print("Eligible clips N = " + str(n_eligible) + ". n on each row is complete cases.")
    print(
        "Seed = "
        + str(SEED)
        + ". Bootstrap replicates = "
        + str(N_BOOTSTRAP)
        + ". Monte Carlo permutations = "
        + str(N_PERMUTATIONS)
        + ". Exact enumeration when complete cases <= "
        + str(EXACT_MAX_N)
        + "."
    )
    print("Tested dimensions:")
    tested = results.loc[results["analysis_status"].eq("tested")]
    if tested.empty:
        print("(none)")
    for row in tested.itertuples(index=False):
        if row.bootstrap_undefined_replicates:
            print(
                "Warning: "
                + row.dimension
                + " had "
                + str(int(row.bootstrap_undefined_replicates))
                + " undefined bootstrap replicates out of "
                + str(int(row.bootstrap_attempted_replicates))
                + "."
            )
        if row.bootstrap_valid_replicates < MIN_VALID_BOOTSTRAPS:
            print(
                "Warning: "
                + row.dimension
                + " bootstrap interval is unavailable ("
                + str(int(row.bootstrap_valid_replicates))
                + " valid replicates; at least "
                + str(MIN_VALID_BOOTSTRAPS)
                + " are required)."
            )
        print(
            row.dimension
            + "  n="
            + str(int(row.n_complete_cases))
            + "  missing="
            + str(int(row.n_missing_metric))
            + "/"
            + str(n_eligible)
            + "  kendall_tau_b="
            + fmt(row.kendall_tau_b, 3)
            + "  CI=["
            + fmt(row.ci_95_lower, 3)
            + ", "
            + fmt(row.ci_95_upper, 3)
            + "]  p_permutation_raw="
            + fmt(row.p_permutation_raw, 4)
            + "  p_permutation_holm="
            + fmt(row.p_permutation_holm, 4)
            + "  kendall_tau_b_median="
            + fmt(row.kendall_tau_b_median, 3)
            + "  permutation="
            + row.permutation_method
            + "  bootstrap_valid="
            + str(int(row.bootstrap_valid_replicates))
            + "  bootstrap_undefined="
            + str(int(row.bootstrap_undefined_replicates))
        )
    print("Skipped dimensions:")
    skipped = results.loc[~results["analysis_status"].eq("tested")]
    if skipped.empty:
        print("(none)")
    for row in skipped.itertuples(index=False):
        print(
            row.dimension
            + "  n="
            + str(int(row.n_complete_cases))
            + "  missing="
            + str(int(row.n_missing_metric))
            + "/"
            + str(n_eligible)
            + "  status="
            + row.analysis_status
        )
    print(
        "Holm family size = "
        + str(len(DIMENSIONS))
        + ". Untestable prespecified dimensions enter the correction as p = 1, "
        + "and their coefficient and p-values are reported as unavailable. "
        + "CIs are marginal paired-clip bootstrap 95% percentile intervals, "
        + "not simultaneous family-wide intervals. "
        + "They condition on the observed viewer panel. "
        + "p_permutation_raw is a two-sided exact or Monte Carlo permutation p "
        + "and is never zero. "
        + "kendall_tau_b_median repeats tau-b on median enjoyment and is not a second confirmatory test. "
        + "Associations are not causal."
    )


def write_results(results):
    folder = output_dir()
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / RESULTS_NAME
    columns = [
        "dimension",
        "analysis_status",
        "n_eligible_clips",
        "n_complete_cases",
        "n_missing_metric",
        "n_unique_metric_values",
        "n_unique_enjoyment_values",
        "kendall_tau_b",
        "ci_95_lower",
        "ci_95_upper",
        "p_permutation_raw",
        "p_permutation_holm",
        "permutation_method",
        "bootstrap_attempted_replicates",
        "bootstrap_valid_replicates",
        "bootstrap_undefined_replicates",
        "kendall_tau_b_median",
        "reject_holm",
    ]
    results[columns].to_csv(path, index=False)
    print("Wrote " + str(path))


def main():
    clear_prior_results()
    folder = data_dir()
    print("DATA_DIR = " + str(folder))
    metrics = load_metrics(folder)
    if metrics is None:
        return
    ratings = load_ratings(folder)
    if ratings is None:
        return
    ratings = validate_ratings(ratings, metrics)
    included = clip_table(metrics, ratings)
    if len(included) < MIN_CLIPS:
        stop(
            "Kendall's tau-b needs at least "
            + str(MIN_CLIPS)
            + " eligible clips; found "
            + str(len(included))
            + ".",
            "Confirmatory inference was not run.",
        )
        return
    blocked = confirmatory_block(included)
    if blocked:
        stop(*blocked)
        return
    results = analyze(included)
    print_results(results)
    write_results(results)


main()
